# Round 3 — results

**Question.** Training a recurrent student (DynVision DyRCNNx4) to recognize objects *and* to match the internal representations of a target network (CKA loss): what does aligning **which layers**, at **which timesteps**, and **how strongly** do to the student's representations and behavior?

**Models**
- **Target**: DyRCNNx4 trained by us on the ground-truth Imagenette labels (frozen afterwards). It plays the role of "the brain": students learn its predictions and are aligned to its activity.
- **Students**: same architecture as the target, trained on the target's final predictions (pseudo-labels) with
  `loss = CE + Σ weight × log(1 − CKA(student layer, target layer))`.

| Condition | Alignment term(s) | Total weight |
|---|---|---|
| Control | none (CE only) | 0 |
| IT, w=1 | IT at t19 | 1.0 |
| IT, w=0.1 | IT at t19 | 0.1 |
| V1+V4+IT, 0.1 each | V1, V4, IT at t19 | 0.3 |
| V1+V4+IT, 0.1 total | V1, V4, IT at t19, 0.033 each | 0.1 |
| IT at 4 timesteps, 0.1 total | IT at t2, t6, t12, t19, 0.025 each | 0.1 |

Each condition is trained with **3 seeds** (2723, 2724, 2725). Within a seed, all conditions start from the same weights and see the same batches in the same order, so they differ only in the loss (paired design).

**How the figures show the seeds.** Thick line = mean over the 3 seeds; thin faint lines = the individual seeds. Every condition has the same color in every figure.

**Two kinds of x-axis**
- *Epochs completed* (training curves, validation set): **0 = before training**, 1 = after the first epoch, …, 200 = end.
- *Timestep* (test set, 3,925 images never used for training): the network sees the same image at every one of the 20 timesteps (0–19); every quantity is measured at each timestep. Grey shading = timesteps 0–3, which are not in the CE loss.

**CKA** is always linear CKA (biased estimator) between the student's and the target's activations of the same layer at the same timestep. On the test set it is computed over all 3,925 images at once; during training it is averaged over batches of 32 images (higher than the full-set value).

**Sections**: 0 Target · 1 Summary table · 2 Which layers · 3 Which timesteps · 4 How strongly · Appendix: training

## Settings

Only this cell needs editing.
- **On burst**: `RAW_DIR` = the round 3 output folder. The first run builds the data tables from the raw outputs and saves them in `DATA_DIR`.
- **Anywhere else** (e.g. after downloading `figure_data/`): set `RAW_DIR = None` and `DATA_DIR` to the downloaded folder. Then only the tables are read.

In [ ]:
RAW_DIR = "/scratch/wz2723/dynvision/runs/round3"                 # None = only read DATA_DIR
DATA_DIR = "/scratch/wz2723/dynvision/runs/round3/figure_data"     # the 5 tables + target model card
SAVE_FIGURES = False                                               # True: also write PNG + PDF of every figure
FIG_DIR = "/scratch/wz2723/dynvision/runs/round3/figures"

In [ ]:
import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle

# One color and one display name per condition -- used in EVERY figure.
# Change a color here and it changes everywhere.
COLORS = {
    "control":            "#e8743b",   # orange
    "IT_w1":              "#1f4e9c",   # dark blue
    "IT_w0p1":            "#5b9bd5",   # light blue
    "V1V4IT_w0p1":        "#2e7d32",   # dark green
    "V1V4IT_total0p1":    "#8bc34a",   # light green
    "IT_multiT_total0p1": "#8e44ad",   # purple
}
LABELS = {
    "control":            "Control (CE only)",
    "IT_w1":              "IT, w=1",
    "IT_w0p1":            "IT, w=0.1",
    "V1V4IT_w0p1":        "V1+V4+IT, 0.1 each",
    "V1V4IT_total0p1":    "V1+V4+IT, 0.1 total",
    "IT_multiT_total0p1": "IT at t2/6/12/19, 0.1 total",
}
ORDER = list(COLORS)
TARGET_COLOR = "#555555"
BASELINE_COLOR = "#c9c9c9"

T = 20               # timesteps 0..19
LAST_T = 19          # pseudo-labels and the usual alignment timestep
CE_FIRST_T = 4       # CE is computed on timesteps 4..19
D_TIMESTEPS = [2, 6, 12, 19]   # timesteps aligned in "IT at t2/6/12/19"


def save(fig, name):
    if SAVE_FIGURES:
        Path(FIG_DIR).mkdir(parents=True, exist_ok=True)
        fig.savefig(Path(FIG_DIR) / f"{name}.png", dpi=200, bbox_inches="tight")
        fig.savefig(Path(FIG_DIR) / f"{name}.pdf", bbox_inches="tight")

## Data tables

Built from the raw outputs (only when `RAW_DIR` is set) and saved in `DATA_DIR`. Every figure below is drawn from these tables only.

| File | One row = | Columns |
|---|---|---|
| `target_training.csv` | target, one epoch | `epochs_completed`, `train_ce`, `val_ce`, `train_accuracy_eval` (every 10 epochs), `val_accuracy`, `val_acc_t00`…`val_acc_t19` |
| `student_training.csv` | student (condition, seed), one epoch | `val_ce`, `val_total_loss`, `val_same_answer_as_target`, `val_accuracy`, `val_cka_<layer>_t<tt>` (batch CKA), `grad_ce`, `grad_cka_sum`, `grad_cka_<layer>_t<tt>`, `grad_ratio_cka_to_ce`, `train_grad_norm`, `train_clip_fraction`, `learning_rate` |
| `test_behavior.csv` | model, one timestep (test set) | `model`, `condition`, `seed`, `timestep`, `accuracy`, `same_answer_as_target`, `ce_vs_target_answer` |
| `test_cka.csv` | comparison, layer, timestep (test set) | `comparison` (`student_vs_target` or `control_vs_control`), `condition`, `seed`, `other_seed`, `layer`, `timestep`, `cka` |
| `summary.csv` | condition × seed, timestep 19 (test set) | `accuracy`, `same_answer_as_target`, `ce_vs_target_answer`, `cka_V1`, `cka_V4`, `cka_IT` |

Definitions
- `accuracy`: fraction of images whose predicted class is the true class.
- `same_answer_as_target`: fraction of images whose predicted class equals the **target's final (t19) prediction** — the label the students were trained on. For the target itself it shows how often its earlier timesteps already give its final answer.
- `ce_vs_target_answer`: cross-entropy with respect to that same label.
- `control_vs_control`: CKA between the controls of two different seeds — how similar two students are anyway when only the random seed differs (baseline for the CKA figures).
- `train_ce` (target): mean training loss during each epoch; at epoch 0 the loss of the initial weights on the training set.
- `grad_*`: once per epoch on a fixed batch of 32 training images: the norm of the gradient of each loss term (with its weight); `grad_ratio_cka_to_ce` = |sum of CKA-term gradients| / |CE gradient| (> 1: the alignment terms dominate the update).

In [ ]:
# Build the tables from the raw outputs (skipped when RAW_DIR is None).
if RAW_DIR is not None:
    raw, out = Path(RAW_DIR), Path(DATA_DIR)
    out.mkdir(parents=True, exist_ok=True)

    def split_name(name):                       # "IT_w0p1_s2724" -> ("IT_w0p1", 2724)
        condition, seed = name.rsplit("_s", 1)
        return condition, int(seed)

    # ---- 1. target training curves ------------------------------------------
    m = pd.read_csv(raw / "target_r3/metrics/metrics.csv")
    tt = pd.DataFrame({"epochs_completed": m["epoch"] + 1, "train_ce": m["train_ce_loss"], "val_ce": m["val_ce_loss"],
                       "train_accuracy_eval": m["train_eval_accuracy"], "val_accuracy": m["val_accuracy"]})
    for t in range(T):
        tt[f"val_acc_t{t:02d}"] = m[f"val_acc_t{t:02d}"]
    init = raw / "target_r3/metrics/init_metrics.csv"
    if init.exists():
        i = pd.read_csv(init).iloc[0]
        row0 = {"epochs_completed": 0, "train_ce": i["train_eval_ce_loss"], "val_ce": i["val_ce_loss"],
                "train_accuracy_eval": i["train_eval_accuracy"], "val_accuracy": i["val_accuracy"]}
        row0.update({f"val_acc_t{t:02d}": i[f"val_acc_t{t:02d}"] for t in range(T)})
        tt = pd.concat([pd.DataFrame([row0]), tt], ignore_index=True)
    else:
        print("WARNING: no target init_metrics.csv -> target curves start at epoch 1")
    tt.to_csv(out / "target_training.csv", index=False)
    shutil.copy(raw / "target_r3/model_card.yaml", out / "target_model_card.yaml")

    # ---- 2. student training curves -----------------------------------------
    rename = {"val_ce_loss": "val_ce", "val_total_loss": "val_total_loss",
              "val_target_agreement": "val_same_answer_as_target", "val_gt_accuracy": "val_accuracy",
              "diag_grad_ce": "grad_ce", "diag_grad_cka_sum": "grad_cka_sum",
              "diag_ratio_cka_to_ce": "grad_ratio_cka_to_ce", "train_grad_norm": "train_grad_norm",
              "train_clip_fraction": "train_clip_fraction", "learning_rate": "learning_rate"}
    parts = []
    for run in sorted((raw / "students").iterdir()):
        if not (run / "metrics/metrics.csv").exists():
            continue
        condition, seed = split_name(run.name)
        m = pd.read_csv(run / "metrics/metrics.csv")
        init = run / "metrics/init_metrics.csv"
        if init.exists():
            m = pd.concat([pd.read_csv(init), m], ignore_index=True)
        else:
            print(f"WARNING: no init_metrics.csv in {run.name} -> its curves start at epoch 1")
        cols = {k: v for k, v in rename.items() if k in m.columns}
        cols.update({c: c for c in m.columns if c.startswith("val_cka_")})
        cols.update({c: c.replace("diag_grad_cka_", "grad_cka_") for c in m.columns
                     if c.startswith("diag_grad_cka_") and c != "diag_grad_cka_sum"})
        p = m[list(cols)].rename(columns=cols)
        p.insert(0, "epochs_completed", m["epoch"] + 1)
        p.insert(0, "seed", seed)
        p.insert(0, "condition", condition)
        parts.append(p)
    pd.concat(parts, ignore_index=True).to_csv(out / "student_training.csv", index=False)

    # ---- 3. + 4. test set: behavior and CKA vs target -----------------------
    behavior, cka_parts, have_target = [], [], False
    for e in sorted((raw / "eval").glob("seed*")):
        m = pd.read_csv(e / "metrics_over_time.csv")
        for model, d in m.groupby("model"):
            if model == "target":
                if have_target:                 # identical in every evaluation: keep it once
                    continue
                condition, seed, have_target = "target", np.nan, True
            else:
                condition, seed = split_name(model)
            behavior.append(pd.DataFrame({"model": model, "condition": condition, "seed": seed,
                                          "timestep": d["timestep"], "accuracy": d["gt_accuracy"],
                                          "same_answer_as_target": d["agreement"],
                                          "ce_vs_target_answer": d["ce_pseudo"]}))
        c = pd.read_csv(e / "cka_over_time.csv")
        c = c[c["model_b"] == "target"]
        for model, d in c.groupby("model_a"):
            condition, seed = split_name(model)
            cka_parts.append(pd.DataFrame({"comparison": "student_vs_target", "condition": condition, "seed": seed,
                                           "other_seed": np.nan, "layer": d["layer"], "timestep": d["timestep"],
                                           "cka": d["cka_biased"]}))
    cross = raw / "eval/controls_across_seeds/cka_over_time.csv"
    if cross.exists():
        c = pd.read_csv(cross)
        c = c[(c["model_a"] != "target") & (c["model_b"] != "target")]
        for (a, b), d in c.groupby(["model_a", "model_b"]):
            cka_parts.append(pd.DataFrame({"comparison": "control_vs_control", "condition": "control",
                                           "seed": split_name(a)[1], "other_seed": split_name(b)[1],
                                           "layer": d["layer"], "timestep": d["timestep"], "cka": d["cka_biased"]}))
    else:
        print("WARNING: no controls_across_seeds evaluation -> no seed baseline in the CKA figures")
    test_behavior = pd.concat(behavior, ignore_index=True)
    test_cka = pd.concat(cka_parts, ignore_index=True)
    test_behavior.to_csv(out / "test_behavior.csv", index=False)
    test_cka.to_csv(out / "test_cka.csv", index=False)

    # ---- 5. summary at the last timestep ------------------------------------
    b = test_behavior[test_behavior["timestep"] == LAST_T]
    c = test_cka[(test_cka["comparison"] == "student_vs_target") & (test_cka["timestep"] == LAST_T)]
    c = c.pivot_table(index=["condition", "seed"], columns="layer", values="cka").add_prefix("cka_").reset_index()
    s = b.drop(columns=["model", "timestep"]).merge(c, on=["condition", "seed"], how="left")
    s.to_csv(out / "summary.csv", index=False)

    print("Tables written to", out)
    for f in sorted(out.glob("*.csv")):
        print(f"  {f.name:24s} {len(pd.read_csv(f)):6d} rows")

In [ ]:
# Read the tables (every figure below uses only these).
data = Path(DATA_DIR)
target_training = pd.read_csv(data / "target_training.csv")
student_training = pd.read_csv(data / "student_training.csv")
test_behavior = pd.read_csv(data / "test_behavior.csv")
test_cka = pd.read_csv(data / "test_cka.csv")
summary = pd.read_csv(data / "summary.csv")

try:
    import yaml
    target_card = yaml.safe_load((data / "target_model_card.yaml").read_text())
    TARGET_BEST_EPOCH = target_card["checkpoints"]["best.pt"]["epoch"] + 1   # in epochs completed
except Exception as exc:                                                     # pyyaml missing, or no card
    target_card, TARGET_BEST_EPOCH = None, None
    print("target model card not read:", exc)

print("seeds per condition:", student_training.groupby("condition")["seed"].nunique().to_dict())
print("epoch 0 (before training) present:", bool((student_training["epochs_completed"] == 0).any()))
print("target best checkpoint (epochs completed):", TARGET_BEST_EPOCH)

## 0 — The target

The students learn the target's predictions and are aligned to its activity, so everything below is relative to this network. Its full configuration is in `target_model_card.yaml`.

### Figure 0.1 — Target: training and validation loss

- **Question**: did the target train normally, and is the checkpoint used as the target a sensible one?
- **Plotted**: `target_training.csv` — `train_ce` (dashed) and `val_ce` (solid) vs `epochs_completed`. Vertical line: the checkpoint used as the target (lowest validation loss).
- **How to read**: a validation curve that flattens while the training curve keeps falling = the target starts fitting its training images specifically (no data augmentation was used).

In [ ]:
d = target_training
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(d["epochs_completed"], d["train_ce"], color="#8c8c8c", lw=1.6, ls="--", label="training set")
ax.plot(d["epochs_completed"], d["val_ce"], color=TARGET_COLOR, lw=2, label="validation set")
if TARGET_BEST_EPOCH is not None:
    ax.axvline(TARGET_BEST_EPOCH, color="#b22222", lw=1, ls=":", label=f"checkpoint used ({TARGET_BEST_EPOCH} epochs)")
ax.set_xlabel("epochs completed (0 = before training)")
ax.set_ylabel("cross-entropy vs true label")
ax.set_title("Target: loss during training")
ax.legend(frameon=False)
plt.tight_layout()
save(fig, "fig0_1_target_loss")
plt.show()

### Figure 0.2 — Target: accuracy on training vs validation images

- **Question**: how much does the target memorize its training images?
- **Plotted**: `target_training.csv` — `train_accuracy_eval` (dots, measured every 10 epochs) and `val_accuracy` vs `epochs_completed`; prediction at t19.
- **How to read**: the gap between the two = memorization. It matters because the students are trained on the same training images: the larger the gap, the closer the target's answers on those images are to the true labels.

In [ ]:
d = target_training
tr = d.dropna(subset=["train_accuracy_eval"])
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(tr["epochs_completed"], tr["train_accuracy_eval"], color="#8c8c8c", lw=1.4, marker="o", ms=4, label="training set")
ax.plot(d["epochs_completed"], d["val_accuracy"], color=TARGET_COLOR, lw=2, label="validation set")
if TARGET_BEST_EPOCH is not None:
    ax.axvline(TARGET_BEST_EPOCH, color="#b22222", lw=1, ls=":", label="checkpoint used")
ax.axhline(0.1, color="#bbbbbb", lw=1, ls="--", label="chance (10 classes)")
ax.set_ylim(0, 1)
ax.set_xlabel("epochs completed (0 = before training)")
ax.set_ylabel("accuracy (prediction at t19)")
ax.set_title("Target: accuracy during training")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
save(fig, "fig0_2_target_accuracy")
plt.show()

### Figure 0.3 — Target: accuracy at each timestep (test set)

- **Question**: how does the target's recognition develop over the 20 timesteps of one image presentation?
- **Plotted**: `test_behavior.csv`, `model == "target"` — `accuracy` vs `timestep`.
- **How to read**: the recurrent dynamics at work — how fast the target reaches its final answer. Its answer at t19 is the label the students learn.

In [ ]:
d = test_behavior[test_behavior["model"] == "target"]
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.axvspan(-0.5, CE_FIRST_T - 0.5, color="#f2f2f2", zorder=0, label="not in the loss (t0-t3)")
ax.plot(d["timestep"], d["accuracy"], color=TARGET_COLOR, lw=2, marker="o", ms=4, label="target")
ax.axhline(0.1, color="#bbbbbb", lw=1, ls="--", label="chance")
ax.set_ylim(0, 1)
ax.set_xticks(range(0, T, 2))
ax.set_xlabel("timestep")
ax.set_ylabel("test accuracy")
ax.set_title("Target: accuracy across timesteps")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
save(fig, "fig0_3_target_accuracy_by_timestep")
plt.show()

## 1 — Summary table (test set, timestep 19)

Mean ± standard deviation over the 3 seeds. Source: `summary.csv`.

In [ ]:
cols = ["accuracy", "same_answer_as_target", "ce_vs_target_answer", "cka_V1", "cka_V4", "cka_IT"]
s = summary[summary["condition"] != "target"]
mean = s.groupby("condition")[cols].mean().reindex(ORDER)
std = s.groupby("condition")[cols].std().reindex(ORDER)
table = mean.round(3).astype(str) + " ± " + std.round(3).astype(str)
table.index = [LABELS[c] for c in table.index]
target_acc = summary.loc[summary["condition"] == "target", "accuracy"]
print(f"Target accuracy at t19: {target_acc.iloc[0]:.3f}" if len(target_acc) else "")
table

## 2 — Which layers are aligned

Conditions compared: control, **IT** (w=0.1), **V1+V4+IT with the same total weight** (0.1), and V1+V4+IT with 0.1 each (0.3 total). "IT, w=0.1" vs "V1+V4+IT, 0.1 total" differ only in *where* the alignment is applied.

### Figure 2.1 — IT: similarity to the target at each timestep (test set)

- **Question**: how close is each student's IT to the target's IT, and at which timesteps?
- **Plotted**: `test_cka.csv`, `comparison == "student_vs_target"`, `layer == "IT"` — `cka` vs `timestep`. Grey band: `comparison == "control_vs_control"` (range over the 3 pairs of control seeds).
- **How to read**: the alignment terms act at t19 (dotted line); a gap at earlier timesteps means the alignment spread through the dynamics. Values above the grey band: the student is closer to the target than two independently seeded controls are to each other.

In [ ]:
LAYER = "IT"
CONDITIONS = ["control", "IT_w0p1", "V1V4IT_total0p1", "V1V4IT_w0p1"]

d = test_cka[(test_cka["comparison"] == "student_vs_target") & (test_cka["layer"] == LAYER)]
base = test_cka[(test_cka["comparison"] == "control_vs_control") & (test_cka["layer"] == LAYER)]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.axvspan(-0.5, CE_FIRST_T - 0.5, color="#f2f2f2", zorder=0)
if len(base):
    b = base.groupby("timestep")["cka"]
    ax.fill_between(b.min().index, b.min(), b.max(), color=BASELINE_COLOR, alpha=0.6, lw=0,
                    label="control vs control (different seeds)")
for cond in CONDITIONS:
    dc = d[d["condition"] == cond]
    for seed, ds in dc.groupby("seed"):
        ax.plot(ds["timestep"], ds["cka"], color=COLORS[cond], lw=0.8, alpha=0.35)
    mean = dc.groupby("timestep")["cka"].mean()
    ax.plot(mean.index, mean.values, color=COLORS[cond], lw=2.2, marker="o", ms=3, label=LABELS[cond])
ax.axvline(LAST_T, color="#999999", lw=1, ls=":")
ax.set_xticks(range(0, T, 2))
ax.set_xlabel("timestep")
ax.set_ylabel(f"CKA with the target ({LAYER})")
ax.set_title(f"{LAYER}: similarity to the target across timesteps")
ax.legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
save(fig, f"fig2_1_{LAYER}_cka_by_timestep")
plt.show()

### Figure 2.2 — V4: similarity to the target at each timestep (test set)

- **Question**: how close is each student's V4 to the target's V4, and at which timesteps?
- **Plotted**: `test_cka.csv`, `comparison == "student_vs_target"`, `layer == "V4"` — `cka` vs `timestep`. Grey band: `comparison == "control_vs_control"` (range over the 3 pairs of control seeds).
- **How to read**: the alignment terms act at t19 (dotted line); a gap at earlier timesteps means the alignment spread through the dynamics. Values above the grey band: the student is closer to the target than two independently seeded controls are to each other.

In [ ]:
LAYER = "V4"
CONDITIONS = ["control", "IT_w0p1", "V1V4IT_total0p1", "V1V4IT_w0p1"]

d = test_cka[(test_cka["comparison"] == "student_vs_target") & (test_cka["layer"] == LAYER)]
base = test_cka[(test_cka["comparison"] == "control_vs_control") & (test_cka["layer"] == LAYER)]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.axvspan(-0.5, CE_FIRST_T - 0.5, color="#f2f2f2", zorder=0)
if len(base):
    b = base.groupby("timestep")["cka"]
    ax.fill_between(b.min().index, b.min(), b.max(), color=BASELINE_COLOR, alpha=0.6, lw=0,
                    label="control vs control (different seeds)")
for cond in CONDITIONS:
    dc = d[d["condition"] == cond]
    for seed, ds in dc.groupby("seed"):
        ax.plot(ds["timestep"], ds["cka"], color=COLORS[cond], lw=0.8, alpha=0.35)
    mean = dc.groupby("timestep")["cka"].mean()
    ax.plot(mean.index, mean.values, color=COLORS[cond], lw=2.2, marker="o", ms=3, label=LABELS[cond])
ax.axvline(LAST_T, color="#999999", lw=1, ls=":")
ax.set_xticks(range(0, T, 2))
ax.set_xlabel("timestep")
ax.set_ylabel(f"CKA with the target ({LAYER})")
ax.set_title(f"{LAYER}: similarity to the target across timesteps")
ax.legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
save(fig, f"fig2_2_{LAYER}_cka_by_timestep")
plt.show()

### Figure 2.3 — V1: similarity to the target at each timestep (test set)

- **Question**: how close is each student's V1 to the target's V1, and at which timesteps?
- **Plotted**: `test_cka.csv`, `comparison == "student_vs_target"`, `layer == "V1"` — `cka` vs `timestep`. Grey band: `comparison == "control_vs_control"` (range over the 3 pairs of control seeds).
- **How to read**: the alignment terms act at t19 (dotted line); a gap at earlier timesteps means the alignment spread through the dynamics. Values above the grey band: the student is closer to the target than two independently seeded controls are to each other.

In [ ]:
LAYER = "V1"
CONDITIONS = ["control", "IT_w0p1", "V1V4IT_total0p1", "V1V4IT_w0p1"]

d = test_cka[(test_cka["comparison"] == "student_vs_target") & (test_cka["layer"] == LAYER)]
base = test_cka[(test_cka["comparison"] == "control_vs_control") & (test_cka["layer"] == LAYER)]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.axvspan(-0.5, CE_FIRST_T - 0.5, color="#f2f2f2", zorder=0)
if len(base):
    b = base.groupby("timestep")["cka"]
    ax.fill_between(b.min().index, b.min(), b.max(), color=BASELINE_COLOR, alpha=0.6, lw=0,
                    label="control vs control (different seeds)")
for cond in CONDITIONS:
    dc = d[d["condition"] == cond]
    for seed, ds in dc.groupby("seed"):
        ax.plot(ds["timestep"], ds["cka"], color=COLORS[cond], lw=0.8, alpha=0.35)
    mean = dc.groupby("timestep")["cka"].mean()
    ax.plot(mean.index, mean.values, color=COLORS[cond], lw=2.2, marker="o", ms=3, label=LABELS[cond])
ax.axvline(LAST_T, color="#999999", lw=1, ls=":")
ax.set_xticks(range(0, T, 2))
ax.set_xlabel("timestep")
ax.set_ylabel(f"CKA with the target ({LAYER})")
ax.set_title(f"{LAYER}: similarity to the target across timesteps")
ax.legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
save(fig, f"fig2_3_{LAYER}_cka_by_timestep")
plt.show()

### Figure 2.4 — How much each layer moved toward the target (t19, paired by seed)

- **Question**: when we align some layers, do the *other* layers move toward the target too, or away from it?
- **Plotted**: `test_cka.csv`, `comparison == "student_vs_target"`, `timestep == 19` — for each seed: CKA of the condition **minus** CKA of the control of the same seed. Small dots = seeds; horizontal bar = mean.
- **How to read**: above 0 = closer to the target than the control; below 0 = further away. For "IT, w=0.1", V1 and V4 are not in the loss.

In [ ]:
CONDITIONS = ["IT_w0p1", "V1V4IT_total0p1", "V1V4IT_w0p1"]
LAYERS = ["V1", "V4", "IT"]

d = test_cka[(test_cka["comparison"] == "student_vs_target") & (test_cka["timestep"] == LAST_T)]
control = d[d["condition"] == "control"].set_index(["seed", "layer"])["cka"]
fig, ax = plt.subplots(figsize=(6.5, 4.2))
width = 0.25
for i, cond in enumerate(CONDITIONS):
    delta = (d[d["condition"] == cond].set_index(["seed", "layer"])["cka"] - control).dropna().reset_index()
    for j, layer in enumerate(LAYERS):
        x = j + (i - (len(CONDITIONS) - 1) / 2) * width
        v = delta.loc[delta["layer"] == layer, "cka"]
        ax.scatter(np.full(len(v), x), v, color=COLORS[cond], s=20, alpha=0.6)
        ax.scatter([x], [v.mean()], color=COLORS[cond], marker="_", s=300, linewidths=3,
                   label=LABELS[cond] if j == 0 else None)
ax.axhline(0, color="#999999", lw=1)
ax.set_xticks(range(len(LAYERS)))
ax.set_xticklabels(LAYERS)
ax.set_xlabel("layer")
ax.set_ylabel("CKA with the target:\ncondition − control (same seed)")
ax.set_title("How much each layer moved toward the target (t19)")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
save(fig, "fig2_4_delta_cka_by_layer")
plt.show()

### Figure 2.5 — How often the student gives the target's final answer (test set)

- **Question**: does aligning more layers also make the students *behave* more like the target, and at which timesteps?
- **Plotted**: `test_behavior.csv` — `same_answer_as_target` vs `timestep`: fraction of test images where the student's prediction at that timestep equals the target's prediction at t19. Dashed grey: the target itself (how often its earlier timesteps already give its final answer).
- **How to read**: higher = closer to the target's behavior. The shape over timesteps shows whether the student reaches the answer as early or as late as the target.

In [ ]:
CONDITIONS = ["control", "IT_w0p1", "V1V4IT_total0p1", "V1V4IT_w0p1"]

d = test_behavior
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.axvspan(-0.5, CE_FIRST_T - 0.5, color="#f2f2f2", zorder=0)
tgt = d[d["model"] == "target"]
ax.plot(tgt["timestep"], tgt["same_answer_as_target"], color=TARGET_COLOR, lw=1.6, ls="--", label="target itself")
for cond in CONDITIONS:
    dc = d[d["condition"] == cond]
    for seed, ds in dc.groupby("seed"):
        ax.plot(ds["timestep"], ds["same_answer_as_target"], color=COLORS[cond], lw=0.8, alpha=0.35)
    mean = dc.groupby("timestep")["same_answer_as_target"].mean()
    ax.plot(mean.index, mean.values, color=COLORS[cond], lw=2.2, marker="o", ms=3, label=LABELS[cond])
ax.set_ylim(0, 1.02)
ax.set_xticks(range(0, T, 2))
ax.set_xlabel("timestep")
ax.set_ylabel("same answer as the target's final answer\n(fraction of test images)")
ax.set_title("How often the student gives the target's final answer")
ax.legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
save(fig, "fig2_5_same_answer_layers")
plt.show()

## 3 — Which timesteps are aligned

Conditions compared: control, **IT at t19** (w=0.1), **IT at t2, t6, t12 and t19** (0.025 each, same total weight 0.1). The two aligned conditions differ only in *when* the alignment is applied.

### Figure 3.1 — IT: similarity to the target at each timestep (test set)

- **Question**: does aligning IT at several timesteps keep it close to the target throughout the dynamics, compared with aligning it only at the end?
- **Plotted**: `test_cka.csv`, `comparison == "student_vs_target"`, `layer == "IT"` — `cka` vs `timestep`. Purple dotted lines: the four aligned timesteps (t19 is also the only one for "IT, w=0.1"). Grey band: control vs control.
- **How to read**: compare the two aligned conditions between the aligned timesteps — does the multi-timestep student stay high in between, or only at the four constrained points?

In [ ]:
LAYER = "IT"
CONDITIONS = ["control", "IT_w0p1", "IT_multiT_total0p1"]

d = test_cka[(test_cka["comparison"] == "student_vs_target") & (test_cka["layer"] == LAYER)]
base = test_cka[(test_cka["comparison"] == "control_vs_control") & (test_cka["layer"] == LAYER)]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.axvspan(-0.5, CE_FIRST_T - 0.5, color="#f2f2f2", zorder=0)
if len(base):
    b = base.groupby("timestep")["cka"]
    ax.fill_between(b.min().index, b.min(), b.max(), color=BASELINE_COLOR, alpha=0.6, lw=0,
                    label="control vs control (different seeds)")
for t in D_TIMESTEPS:
    ax.axvline(t, color=COLORS["IT_multiT_total0p1"], lw=1, ls=":", alpha=0.7)
for cond in CONDITIONS:
    dc = d[d["condition"] == cond]
    for seed, ds in dc.groupby("seed"):
        ax.plot(ds["timestep"], ds["cka"], color=COLORS[cond], lw=0.8, alpha=0.35)
    mean = dc.groupby("timestep")["cka"].mean()
    ax.plot(mean.index, mean.values, color=COLORS[cond], lw=2.2, marker="o", ms=3, label=LABELS[cond])
ax.set_xticks(range(0, T, 2))
ax.set_xlabel("timestep")
ax.set_ylabel("CKA with the target (IT)")
ax.set_title("IT: aligning at one vs four timesteps")
ax.legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
save(fig, "fig3_1_IT_cka_one_vs_four_timesteps")
plt.show()

### Figure 3.2 — Where in time each condition's alignment lands (IT, heatmap)

- **Question**: for every condition at once — at which timesteps did IT move toward the target?
- **Plotted**: `test_cka.csv`, `comparison == "student_vs_target"`, `layer == "IT"` — each cell: CKA of the condition minus CKA of the control of the same seed, averaged over the 3 seeds. Rows = conditions, columns = timesteps. Black boxes: the timesteps where IT is in that condition's loss.
- **How to read**: red = closer to the target than the control, blue = further away, white = no difference.

In [ ]:
LAYER = "IT"
CONDITIONS = ["IT_w1", "IT_w0p1", "V1V4IT_w0p1", "V1V4IT_total0p1", "IT_multiT_total0p1"]
ALIGNED_TIMESTEPS = {"IT_w1": [19], "IT_w0p1": [19], "V1V4IT_w0p1": [19], "V1V4IT_total0p1": [19],
                     "IT_multiT_total0p1": D_TIMESTEPS}

d = test_cka[(test_cka["comparison"] == "student_vs_target") & (test_cka["layer"] == LAYER)]
control = d[d["condition"] == "control"].set_index(["seed", "timestep"])["cka"]
M = np.full((len(CONDITIONS), T), np.nan)
for i, cond in enumerate(CONDITIONS):
    delta = d[d["condition"] == cond].set_index(["seed", "timestep"])["cka"] - control
    mean = delta.groupby(level="timestep").mean()
    M[i, mean.index.astype(int)] = mean.values
lim = np.nanmax(np.abs(M))

fig, ax = plt.subplots(figsize=(9, 3.6))
im = ax.imshow(M, aspect="auto", cmap="RdBu_r", norm=TwoSlopeNorm(vcenter=0, vmin=-lim, vmax=lim))
for i, cond in enumerate(CONDITIONS):
    for t in ALIGNED_TIMESTEPS[cond]:
        ax.add_patch(Rectangle((t - 0.5, i - 0.5), 1, 1, fill=False, edgecolor="black", lw=1.3))
ax.set_yticks(range(len(CONDITIONS)))
ax.set_yticklabels([LABELS[c] for c in CONDITIONS])
ax.set_xticks(range(T))
ax.set_xlabel("timestep")
ax.set_title("IT: CKA with the target, condition − control (mean of 3 seeds)")
fig.colorbar(im, ax=ax, label="Δ CKA vs control")
plt.tight_layout()
save(fig, "fig3_2_heatmap_delta_IT_cka")
plt.show()

### Figure 3.3 — How often the student gives the target's final answer (test set)

- **Question**: does aligning IT at several timesteps make the students' behavior over time more target-like?
- **Plotted**: `test_behavior.csv` — `same_answer_as_target` vs `timestep` (definition in Figure 2.5). Dashed grey: the target itself.

In [ ]:
CONDITIONS = ["control", "IT_w0p1", "IT_multiT_total0p1"]

d = test_behavior
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.axvspan(-0.5, CE_FIRST_T - 0.5, color="#f2f2f2", zorder=0)
tgt = d[d["model"] == "target"]
ax.plot(tgt["timestep"], tgt["same_answer_as_target"], color=TARGET_COLOR, lw=1.6, ls="--", label="target itself")
for cond in CONDITIONS:
    dc = d[d["condition"] == cond]
    for seed, ds in dc.groupby("seed"):
        ax.plot(ds["timestep"], ds["same_answer_as_target"], color=COLORS[cond], lw=0.8, alpha=0.35)
    mean = dc.groupby("timestep")["same_answer_as_target"].mean()
    ax.plot(mean.index, mean.values, color=COLORS[cond], lw=2.2, marker="o", ms=3, label=LABELS[cond])
ax.set_ylim(0, 1.02)
ax.set_xticks(range(0, T, 2))
ax.set_xlabel("timestep")
ax.set_ylabel("same answer as the target's final answer\n(fraction of test images)")
ax.set_title("How often the student gives the target's final answer")
ax.legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
save(fig, "fig3_3_same_answer_timesteps")
plt.show()

### Figure 3.4 — Same, as the difference from the control (paired by seed)

- **Plotted**: `test_behavior.csv` — for each seed, `same_answer_as_target` of the condition minus that of the control of the same seed. Thin lines = seeds, thick = mean.
- **How to read**: above 0 = gives the target's answer more often than the control at that timestep. If all thin lines stay on the same side of 0, the effect holds in every seed.

In [ ]:
CONDITIONS = ["IT_w0p1", "IT_multiT_total0p1"]
COLUMN = "same_answer_as_target"

d = test_behavior[test_behavior["condition"] != "target"]
control = d[d["condition"] == "control"].set_index(["seed", "timestep"])[COLUMN]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.axvspan(-0.5, CE_FIRST_T - 0.5, color="#f2f2f2", zorder=0)
ax.axhline(0, color="#999999", lw=1)
for cond in CONDITIONS:
    delta = (d[d["condition"] == cond].set_index(["seed", "timestep"])[COLUMN] - control).reset_index()
    for seed, ds in delta.groupby("seed"):
        ax.plot(ds["timestep"], ds[COLUMN], color=COLORS[cond], lw=0.8, alpha=0.35)
    mean = delta.groupby("timestep")[COLUMN].mean()
    ax.plot(mean.index, mean.values, color=COLORS[cond], lw=2.2, marker="o", ms=3, label=LABELS[cond])
ax.set_xticks(range(0, T, 2))
ax.set_xlabel("timestep")
ax.set_ylabel("same answer as the target:\ncondition − control (same seed)")
ax.set_title("Behavior relative to the control")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
save(fig, "fig3_4_delta_same_answer")
plt.show()

### Figure 3.5 — During training: IT similarity at the four aligned timesteps

- **Question**: in the multi-timestep condition, are some timesteps aligned faster or more easily than others?
- **Plotted**: `student_training.csv`, `condition == "IT_multiT_total0p1"` — `val_cka_IT_t02`, `_t06`, `_t12`, `_t19` vs `epochs_completed` (validation set, CKA averaged over batches of 32 images, so higher than the full test-set values).

In [ ]:
CONDITION = "IT_multiT_total0p1"
COLUMNS = {"val_cka_IT_t02": ("t2", "#d7b5e6"), "val_cka_IT_t06": ("t6", "#b07cc6"),
           "val_cka_IT_t12": ("t12", "#8e44ad"), "val_cka_IT_t19": ("t19", "#4a235a")}

d = student_training[student_training["condition"] == CONDITION]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for col, (label, color) in COLUMNS.items():
    for seed, ds in d.groupby("seed"):
        ax.plot(ds["epochs_completed"], ds[col], color=color, lw=0.7, alpha=0.35)
    mean = d.groupby("epochs_completed")[col].mean()
    ax.plot(mean.index, mean.values, color=color, lw=2, label=f"IT at {label}")
ax.set_xlabel("epochs completed (0 = before training)")
ax.set_ylabel("batch CKA with the target (validation)")
ax.set_title(f"{LABELS[CONDITION]}: alignment of each timestep during training")
ax.legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
save(fig, "fig3_5_training_IT_cka_four_timesteps")
plt.show()

## 4 — How strongly to align

Conditions compared: IT with weight 1 vs 0.1; V1+V4+IT with 0.1 each vs 0.1 in total.

### Figure 4.1 — Alignment vs recognition: is there a cost?

- **Question**: do the students that are most aligned with the target pay for it in accuracy?
- **Plotted**: `summary.csv` (test set, t19) — x: `cka_IT`, y: `accuracy`. Small dots = seeds, large dots = mean of each condition. Dashed line: the target's accuracy.
- **How to read**: points further right are more aligned; points lower are less accurate. A cost would show as a downward trend from left to right.

In [ ]:
CONDITIONS = ORDER
X, Y = "cka_IT", "accuracy"

s = summary
fig, ax = plt.subplots(figsize=(7, 4.6))
for cond in CONDITIONS:
    sc = s[s["condition"] == cond]
    ax.scatter(sc[X], sc[Y], color=COLORS[cond], s=22, alpha=0.5)
    ax.scatter([sc[X].mean()], [sc[Y].mean()], color=COLORS[cond], s=110, edgecolor="white", lw=1, label=LABELS[cond])
tgt = s.loc[s["condition"] == "target", Y]
if len(tgt):
    ax.axhline(tgt.iloc[0], color=TARGET_COLOR, lw=1, ls="--", label="target")
ax.set_xlabel("IT: CKA with the target (t19)")
ax.set_ylabel("test accuracy (t19)")
ax.set_title("Alignment vs recognition")
ax.legend(frameon=False, fontsize=8, loc="best")
plt.tight_layout()
save(fig, "fig4_1_alignment_vs_accuracy")
plt.show()

### Figure 4.2 — During training: loss on the target's answers (IT weight 1 vs 0.1)

- **Question**: does a stronger alignment slow down or limit learning the target's answers?
- **Plotted**: `student_training.csv` — `val_ce` (cross-entropy vs the target's final answer, validation set, averaged over timesteps 4–19) vs `epochs_completed`.

In [ ]:
CONDITIONS = ["control", "IT_w0p1", "IT_w1"]

d = student_training
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for cond in CONDITIONS:
    dc = d[d["condition"] == cond]
    for seed, ds in dc.groupby("seed"):
        ax.plot(ds["epochs_completed"], ds["val_ce"], color=COLORS[cond], lw=0.7, alpha=0.35)
    mean = dc.groupby("epochs_completed")["val_ce"].mean()
    ax.plot(mean.index, mean.values, color=COLORS[cond], lw=2, label=LABELS[cond])
ax.set_xlabel("epochs completed (0 = before training)")
ax.set_ylabel("cross-entropy vs the target's answer\n(validation)")
ax.set_title("Loss on the target's answers during training")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
save(fig, "fig4_2_val_ce")
plt.show()

### Figure 4.3 — During training: loss on the target's answers (V1+V4+IT, 0.1 total vs 0.1 each)

- **Question**: does a stronger alignment slow down or limit learning the target's answers?
- **Plotted**: `student_training.csv` — `val_ce` (cross-entropy vs the target's final answer, validation set, averaged over timesteps 4–19) vs `epochs_completed`.

In [ ]:
CONDITIONS = ["control", "V1V4IT_total0p1", "V1V4IT_w0p1"]

d = student_training
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for cond in CONDITIONS:
    dc = d[d["condition"] == cond]
    for seed, ds in dc.groupby("seed"):
        ax.plot(ds["epochs_completed"], ds["val_ce"], color=COLORS[cond], lw=0.7, alpha=0.35)
    mean = dc.groupby("epochs_completed")["val_ce"].mean()
    ax.plot(mean.index, mean.values, color=COLORS[cond], lw=2, label=LABELS[cond])
ax.set_xlabel("epochs completed (0 = before training)")
ax.set_ylabel("cross-entropy vs the target's answer\n(validation)")
ax.set_title("Loss on the target's answers during training")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
save(fig, "fig4_3_val_ce")
plt.show()

## Appendix — Training details

Checks that training behaved as intended. Not needed to follow the results above.

### Figure A.1 — Loss on the target's answers (control, IT at t19, IT at four timesteps)

Same as Figures 4.2 / 4.3 for the conditions of section 3. Source: `student_training.csv`, `val_ce`.

In [ ]:
CONDITIONS = ["control", "IT_w0p1", "IT_multiT_total0p1"]

d = student_training
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for cond in CONDITIONS:
    dc = d[d["condition"] == cond]
    for seed, ds in dc.groupby("seed"):
        ax.plot(ds["epochs_completed"], ds["val_ce"], color=COLORS[cond], lw=0.7, alpha=0.35)
    mean = dc.groupby("epochs_completed")["val_ce"].mean()
    ax.plot(mean.index, mean.values, color=COLORS[cond], lw=2, label=LABELS[cond])
ax.set_xlabel("epochs completed (0 = before training)")
ax.set_ylabel("cross-entropy vs the target's answer\n(validation)")
ax.set_title("Loss on the target's answers during training")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
save(fig, "figA_1_val_ce")
plt.show()

### Figure A.2 — IT similarity to the target during training (t19)

- **Plotted**: `student_training.csv` — `val_cka_IT_t19` (batch CKA, validation) vs `epochs_completed`, all conditions, mean over seeds only.
- **How to read**: how quickly each condition aligns IT. At epoch 0 all students of one seed are identical, so the curves start from the same point.

In [ ]:
CONDITIONS = ORDER
COLUMN = "val_cka_IT_t19"

d = student_training
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for cond in CONDITIONS:
    mean = d[d["condition"] == cond].groupby("epochs_completed")[COLUMN].mean()
    ax.plot(mean.index, mean.values, color=COLORS[cond], lw=2, label=LABELS[cond])
ax.set_xlabel("epochs completed (0 = before training)")
ax.set_ylabel("batch CKA with the target, IT t19 (validation)")
ax.set_title("IT alignment during training")
ax.legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
save(fig, "figA_2_training_IT_cka")
plt.show()

### Figure A.3 — Balance between the alignment terms and the CE term

- **Plotted**: `student_training.csv` — `grad_ratio_cka_to_ce` vs `epochs_completed` (log scale), aligned conditions, mean over seeds.
- **How to read**: above 1 (dashed) = the alignment terms dominate the parameter update. The gradient of log(1 − CKA) grows like weight / (1 − CKA), so the ratio tends to rise as CKA approaches 1.

In [ ]:
CONDITIONS = [c for c in ORDER if c != "control"]
COLUMN = "grad_ratio_cka_to_ce"

d = student_training
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for cond in CONDITIONS:
    mean = d[d["condition"] == cond].groupby("epochs_completed")[COLUMN].mean()
    ax.plot(mean.index, mean.values, color=COLORS[cond], lw=1.8, label=LABELS[cond])
ax.axhline(1, color="#999999", lw=1, ls="--")
ax.set_yscale("log")
ax.set_xlabel("epochs completed (0 = before training)")
ax.set_ylabel("|alignment gradients| / |CE gradient|")
ax.set_title("Alignment vs CE: relative gradient size")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
save(fig, "figA_3_gradient_ratio")
plt.show()

### Figure A.4 — Gradient size per training step

- **Plotted**: `student_training.csv` — `train_grad_norm` (mean over the steps of each epoch, before clipping) vs `epochs_completed`, mean over seeds. Dashed: the clipping threshold (500).
- **How to read**: all curves far below the threshold = clipping never changed an update (also see `train_clip_fraction`).

In [ ]:
CONDITIONS = ORDER
COLUMN = "train_grad_norm"
CLIP_THRESHOLD = 500

d = student_training
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for cond in CONDITIONS:
    mean = d[d["condition"] == cond].groupby("epochs_completed")[COLUMN].mean()
    ax.plot(mean.index, mean.values, color=COLORS[cond], lw=1.8, label=LABELS[cond])
ax.axhline(CLIP_THRESHOLD, color="#999999", lw=1, ls="--", label="clipping threshold")
ax.set_yscale("log")
ax.set_xlabel("epochs completed")
ax.set_ylabel("gradient norm per step")
ax.set_title("Gradient size during training")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
save(fig, "figA_4_gradient_norm")
plt.show()
print("largest fraction of clipped steps in any epoch of any run:", d["train_clip_fraction"].max())